# Lecture 28 — Multi-Task RL, Contextual Policies, and Meta-RL

This notebook implements core concepts from Lecture 28:

- Contextual / Goal-conditioned policies
- Memory-based Meta-RL (RL^2 / recurrent policies)
- Optimization-based Meta-RL (MAML-style inner-loop adaptation)

It includes runnable PyTorch code, compact training/evaluation skeletons, small unit tests and plotting helpers. The notebook is intended for learning and small-scale experiments (not full production training loops).


## 1) Setup: imports, device and deterministic seeds

This cell installs (if needed) and imports core libraries, sets device and provides a helper to set seeds for reproducibility.


In [ ]:
# Standard imports
import os
import math
import random
import copy

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

# Utilities
from typing import Dict, Tuple, List, Any

# Device & seed helpers
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed: int = 0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# Quick environment check
print(f"Using device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

# Make experiments reproducible by default in this notebook
set_seed(0)


In [ ]:
## 2) Environment & Task Sampler (goal/context wrappers)

# Note: For clean demo we implement tiny env-like wrappers using numpy. For real experiments use gym/gymnasium.

class SimpleGoalEnv:
    """Toy 2D point-goal environment where the goal is part of the context.
    State: (x,y). Action: delta (dx, dy) clipped. Reward: negative L2 distance to goal (sparse success on threshold).
    """
    def __init__(self, goal: np.ndarray = None, goal_radius=0.1, max_step=0.1):
        self.goal = goal if goal is not None else np.zeros(2, dtype=np.float32)
        self.goal_radius = goal_radius
        self.max_step = max_step
        self.reset()

    def reset(self, goal: np.ndarray = None):
        if goal is not None:
            self.goal = goal.astype(np.float32)
        self.state = np.random.uniform(-1.0, 1.0, size=(2,)).astype(np.float32)
        return self._get_obs()

    def _get_obs(self):
        return self.state.copy()

    def step(self, action: np.ndarray):
        action = np.clip(action, -self.max_step, self.max_step)
        self.state = self.state + action
        dist = np.linalg.norm(self.state - self.goal)
        reward = -dist
        done = dist <= self.goal_radius
        return self._get_obs(), float(reward), bool(done), {}


class TaskSampler:
    """Samples goal tasks from a simple distribution (2D uniform in a square)."""
    def __init__(self, scale=0.8):
        self.scale = scale

    def sample(self):
        return np.random.uniform(-self.scale, self.scale, size=(2,)).astype(np.float32)

# Demo of task sampler and environment
sampler = TaskSampler()
goal = sampler.sample()
env = SimpleGoalEnv(goal)
obs = env.reset()
print("Sampled goal:", goal)
print("Initial obs:", obs)


In [ ]:
## 3) Replay Buffer / Trajectory Storage (lightweight)

class TrajectoryStorage:
    """Simple storage for trajectories including context.
    Stores lists of transitions for one rollout. Useful to build batched mini-batches.
    """
    def __init__(self):
        self.clear()

    def push(self, s, a, r, s_next, done, context, prev_a=None, prev_r=0.0):
        self.states.append(np.array(s, copy=True))
        self.actions.append(np.array(a, copy=True))
        self.rewards.append(float(r))
        self.next_states.append(np.array(s_next, copy=True))
        self.dones.append(bool(done))
        self.contexts.append(np.array(context, copy=True))
        self.prev_actions.append(np.array(prev_a, copy=True) if prev_a is not None else None)
        self.prev_rewards.append(float(prev_r))

    def get(self):
        return dict(
            states=np.asarray(self.states),
            actions=np.asarray(self.actions),
            rewards=np.asarray(self.rewards),
            next_states=np.asarray(self.next_states),
            dones=np.asarray(self.dones),
            contexts=np.asarray(self.contexts),
            prev_actions=np.asarray(self.prev_actions, dtype=object),
            prev_rewards=np.asarray(self.prev_rewards),
        )

    def clear(self):
        self.states = []
        self.actions = []
        self.rewards = []
        self.next_states = []
        self.dones = []
        self.contexts = []
        self.prev_actions = []
        self.prev_rewards = []

# Simple demo for storage
buf = TrajectoryStorage()
buf.push(obs, np.array([0.0,0.0]), -0.5, obs, False, goal)
print("Stored states:", buf.get()["states"].shape)


In [ ]:
## 4) Contextual / Goal-Conditioned Policy (discrete & continuous helpers)

class ContextualPolicy(nn.Module):
    """Implements pi(a | s, context); continuous action (Gaussian mean) or discrete logits.
    For simplicity this demo shows a deterministic MLP output which can be turned into logits or gaussian params.
    """
    def __init__(self, state_dim: int, context_dim: int, action_dim: int, hidden_dim: int = 64, continuous=True):
        super(ContextualPolicy, self).__init__()
        self.input_dim = state_dim + context_dim
        self.continuous = continuous
        self.net = nn.Sequential(
            nn.Linear(self.input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        self.head = nn.Linear(hidden_dim, action_dim)
        if continuous:
            # a simple learned log-std for Gaussian (state-independent)
            self.log_std = nn.Parameter(torch.zeros(action_dim))

    def forward(self, state: torch.Tensor, context: torch.Tensor):
        x = torch.cat([state, context], dim=1)
        h = self.net(x)
        mean = self.head(h)
        if self.continuous:
            std = torch.exp(self.log_std)
            return mean, std
        else:
            return mean  # logits

    def sample_action(self, state: torch.Tensor, context: torch.Tensor):
        if self.continuous:
            mean, std = self.forward(state, context)
            dist = torch.distributions.Normal(mean, std)
            a = dist.rsample()
            logp = dist.log_prob(a).sum(dim=1)
            return a, logp
        else:
            logits = self.forward(state, context)
            dist = torch.distributions.Categorical(logits=logits)
            a = dist.sample()
            logp = dist.log_prob(a)
            return a, logp

# Demo (Part 1)
print("--- Part 1: Contextual Policy Demo ---")
state_dim = 2
goal_dim = 2
action_dim = 2
policy = ContextualPolicy(state_dim, goal_dim, action_dim, continuous=True).to(DEVICE)

# Dummy batch
dummy_states = torch.randn(5, state_dim).to(DEVICE)
dummy_goals = torch.randn(5, goal_dim).to(DEVICE)
actions, logp = policy.sample_action(dummy_states, dummy_goals)
print("Action shape:", actions.shape)
print("Log-prob shape:", logp.shape)


In [ ]:
## 5) Training loop skeleton for contextual policies (REINFORCE-style)

def compute_returns(rewards: List[float], gamma=0.99):
    R = 0.0
    returns = []
    for r in reversed(rewards):
        R = r + gamma * R
        returns.insert(0, R)
    return np.array(returns, dtype=np.float32)


def reinforce_update(policy: ContextualPolicy, optimizer: torch.optim.Optimizer, traj: Dict, gamma=0.99):
    states = torch.tensor(traj['states'], dtype=torch.float32, device=DEVICE)
    contexts = torch.tensor(traj['contexts'], dtype=torch.float32, device=DEVICE)
    actions = torch.tensor(traj['actions'], dtype=torch.float32, device=DEVICE)
    rewards = traj['rewards']

    returns = torch.tensor(compute_returns(rewards, gamma), dtype=torch.float32, device=DEVICE)
    # Simple baseline: mean
    advantages = returns - returns.mean()

    # For continuous policy we compute log probs
    mean, std = policy(states, contexts)
    dist = torch.distributions.Normal(mean, std)
    logp = dist.log_prob(actions).sum(dim=1)

    loss = -(logp * advantages).mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

# Quick run of a single REINFORCE step on a synthetic trajectory
optimizer = torch.optim.Adam(policy.parameters(), lr=1e-3)
traj = buf.get()
# create synthetic multi-step trajectory
traj['states'] = np.random.randn(20, 2)
traj['actions'] = np.random.randn(20, 2)
traj['rewards'] = np.random.randn(20).tolist()
traj['contexts'] = np.tile(goal.reshape(1,2), (20,1))

loss = reinforce_update(policy, optimizer, traj)
print("REINFORCE-style loss:", loss)


In [ ]:
## 6) Evaluation & per-task metrics

def evaluate_policy_continuous(policy: ContextualPolicy, env: SimpleGoalEnv, goal: np.ndarray, episodes=10, max_steps=200, deterministic=False):
    returns = []
    successes = 0
    for _ in range(episodes):
        obs = env.reset(goal)
        total_r = 0.0
        for t in range(max_steps):
            s = torch.tensor(obs.reshape(1,-1), dtype=torch.float32, device=DEVICE)
            g = torch.tensor(goal.reshape(1,-1), dtype=torch.float32, device=DEVICE)
            if deterministic:
                mean, _ = policy(s, g)
                a = mean.detach().cpu().numpy()[0]
            else:
                a, _ = policy.sample_action(s, g)
                a = a.detach().cpu().numpy()[0]
            obs, r, done, _ = env.step(a)
            total_r += r
            if done:
                successes += 1
                break
        returns.append(total_r)
    return np.mean(returns), successes / episodes

# Quick evaluation demo
mean_return, success_rate = evaluate_policy_continuous(policy, env, goal, episodes=20)
print(f"Eval mean return: {mean_return:.3f}, success rate: {success_rate:.2f}")


In [ ]:
## 7) Recurrent Meta-RL Agent (RL^2)

class RecurrentMetaRLAgent(nn.Module):
    """RNN/LSTM-based policy that ingests (s_t, a_{t-1}, r_{t-1}) sequences and outputs action logits / values."""
    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = 128):
        super(RecurrentMetaRLAgent, self).__init__()
        self.hidden_dim = hidden_dim
        rnn_input_dim = state_dim + action_dim + 1

        self.lstm = nn.LSTM(rnn_input_dim, hidden_dim, batch_first=True)
        self.actor = nn.Linear(hidden_dim, action_dim)
        self.critic = nn.Linear(hidden_dim, 1)

    def forward(self, state_seq: torch.Tensor, prev_act_seq: torch.Tensor, prev_rew_seq: torch.Tensor, hidden_state=None):
        # state_seq: [B, T, state_dim]
        if prev_rew_seq.dim() == 2:
            prev_rew_seq = prev_rew_seq.unsqueeze(-1)
        rnn_input = torch.cat([state_seq, prev_act_seq, prev_rew_seq], dim=2)
        lstm_out, new_hidden = self.lstm(rnn_input, hidden_state)
        action_logits = self.actor(lstm_out)
        values = self.critic(lstm_out)
        return action_logits, values.squeeze(-1), new_hidden

# Demo RL^2 forward
print("--- Part 2: RL^2 (Recurrent Meta-RL) Demo ---")
state_dim = 2
action_dim = 2
agent = RecurrentMetaRLAgent(state_dim, action_dim).to(DEVICE)

B = 1; T = 8
states = torch.randn(B, T, state_dim, device=DEVICE)
prev_acts = torch.randn(B, T, action_dim, device=DEVICE)
prev_rews = torch.randn(B, T, 1, device=DEVICE)

logits, values, hidden = agent(states, prev_acts, prev_rews, None)
print("Logits shape:", logits.shape)
print("Values shape:", values.shape)
